# SOPR Capitulation - Trailing Stop Strategy

**The Idea:** Let winners run, cut losers short.

- **Stop Loss:** Exit if price drops X% from entry (protect capital)
- **Trailing Stop:** If price goes up, trail the stop Y% below the peak
- **No Profit Cap:** Don't exit just because we hit +20% - keep riding!

This captures big moves while limiting downside.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Ready!")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")

---
## Trailing Stop Backtester

In [ ]:
def backtest_trailing_stop(
    close: pd.Series,
    entries: pd.Series,
    stop_loss: float = 0.10,           # Initial stop loss from entry (e.g., 0.10 = -10%)
    trailing_stop: float = 0.15,       # Trail this far below peak (e.g., 0.15 = 15%)
    min_profit_to_trail: float = 0.05, # Only start trailing after this profit (e.g., 5%)
    max_hold_days: int = 180,          # Maximum days to hold
):
    """
    Backtest with trailing stop that lets winners run.
    
    Logic:
    1. Enter on signal
    2. Set initial stop at entry_price * (1 - stop_loss)
    3. Once profit > min_profit_to_trail, switch to trailing stop
    4. Trailing stop = peak_price * (1 - trailing_stop)
    5. Exit when price crosses below current stop level
    """
    trades = []
    entry_indices = entries[entries].index.tolist()
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = close.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        # Initialize
        peak_price = entry_price
        initial_stop = entry_price * (1 - stop_loss)
        current_stop = initial_stop
        is_trailing = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        peak_reached = 0
        
        # Track stop levels for visualization
        stop_history = []
        
        for j in range(entry_idx + 1, len(close)):
            current_date = close.index[j]
            current_price = close.iloc[j]
            days_held = j - entry_idx
            
            # Update peak
            if current_price > peak_price:
                peak_price = current_price
                peak_reached = (peak_price - entry_price) / entry_price
            
            # Calculate current P&L
            current_pnl = (current_price - entry_price) / entry_price
            
            # Switch to trailing stop once we have enough profit
            if not is_trailing and current_pnl >= min_profit_to_trail:
                is_trailing = True
            
            # Update stop level
            if is_trailing:
                trailing_stop_level = peak_price * (1 - trailing_stop)
                # Only move stop UP, never down
                if trailing_stop_level > current_stop:
                    current_stop = trailing_stop_level
            
            stop_history.append({'date': current_date, 'stop': current_stop, 'price': current_price})
            
            # Check exit conditions
            
            # 1. Stop hit (either initial or trailing)
            if current_price <= current_stop:
                exit_date = current_date
                exit_price = current_stop  # Assume we exit at stop level
                if is_trailing:
                    exit_reason = f'trailing_stop'
                else:
                    exit_reason = f'stop_loss'
                break
            
            # 2. Max hold days
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = f'max_hold_{max_hold_days}d'
                break
        
        # If no exit, close at end
        if exit_date is None:
            exit_date = close.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        # Record trade
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'peak_pnl_pct': peak_reached,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'stop_history': stop_history
        })
        
        # Skip overlapping entries
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

---
## Test Different Parameters

In [ ]:
# Parameter combinations to test
configs = [
    # (stop_loss, trailing_stop, min_profit_to_trail, name)
    (0.08, 0.12, 0.05, 'Tight: SL8% Trail12%'),
    (0.10, 0.15, 0.05, 'Medium: SL10% Trail15%'),
    (0.10, 0.20, 0.05, 'Medium-Wide: SL10% Trail20%'),
    (0.12, 0.18, 0.05, 'Balanced: SL12% Trail18%'),
    (0.15, 0.20, 0.05, 'Wide: SL15% Trail20%'),
    (0.15, 0.25, 0.10, 'Very Wide: SL15% Trail25%'),
    (0.10, 0.15, 0.10, 'Med + Higher Trigger: SL10% Trail15% @10%'),
    (0.10, 0.20, 0.15, 'Med + Late Trail: SL10% Trail20% @15%'),
]

results = []

for sl, ts, mpt, name in configs:
    trades = backtest_trailing_stop(
        close=close,
        entries=entries,
        stop_loss=sl,
        trailing_stop=ts,
        min_profit_to_trail=mpt,
        max_hold_days=180
    )
    
    if len(trades) > 0:
        total_return = (1 + trades['pnl_pct']).prod() - 1
        win_rate = (trades['pnl_pct'] > 0).mean()
        avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
        avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
        
        # How much profit did we capture vs peak?
        avg_peak = trades['peak_pnl_pct'].mean()
        avg_captured = trades['pnl_pct'].mean()
        
        gross_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].sum()
        gross_loss = abs(trades[trades['pnl_pct'] <= 0]['pnl_pct'].sum())
        profit_factor = gross_win / gross_loss if gross_loss > 0 else np.inf
        
        results.append({
            'strategy': name,
            'stop_loss': sl,
            'trail_stop': ts,
            'min_profit': mpt,
            'n_trades': len(trades),
            'total_return': total_return,
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'avg_peak': avg_peak,
            'avg_captured': avg_captured,
            'avg_days': trades['days_held'].mean()
        })

results_df = pd.DataFrame(results).sort_values('total_return', ascending=False)

print("TRAILING STOP STRATEGY COMPARISON")
print("="*120)
display_cols = ['strategy', 'n_trades', 'total_return', 'win_rate', 'avg_win', 'avg_loss', 'profit_factor', 'avg_days']
print(results_df[display_cols].to_string(index=False))

In [ ]:
# Visualize
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Total Return', 'Win Rate', 'Profit Factor', 'Avg Win vs Avg Loss'])

plot_df = results_df.sort_values('total_return', ascending=True)
colors = ['green' if x > 0 else 'red' for x in plot_df['total_return']]

fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['total_return']*100,
                     orientation='h', marker_color=colors), row=1, col=1)
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['win_rate']*100,
                     orientation='h', marker_color='steelblue'), row=1, col=2)
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['profit_factor'].clip(upper=5),
                     orientation='h', marker_color='purple'), row=2, col=1)

# Avg win vs loss
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['avg_win']*100, name='Avg Win',
                     orientation='h', marker_color='green'), row=2, col=2)
fig.add_trace(go.Bar(y=plot_df['strategy'], x=plot_df['avg_loss']*100, name='Avg Loss',
                     orientation='h', marker_color='red'), row=2, col=2)

fig.add_vline(x=0, line_dash='dash', row=1, col=1)
fig.add_vline(x=50, line_dash='dash', row=1, col=2)
fig.add_vline(x=1, line_dash='dash', row=2, col=1)

fig.update_layout(height=800, showlegend=False, title_text='Trailing Stop Strategies')
fig.show()

In [ ]:
# Best strategy
best = results_df.iloc[0]
print(f"\n{'='*60}")
print(f"BEST STRATEGY: {best['strategy']}")
print(f"{'='*60}")
print(f"Stop Loss: {best['stop_loss']*100:.0f}%")
print(f"Trailing Stop: {best['trail_stop']*100:.0f}%")
print(f"Start Trail After: {best['min_profit']*100:.0f}% profit")
print(f"\nTotal Return: {best['total_return']*100:.1f}%")
print(f"Win Rate: {best['win_rate']*100:.0f}%")
print(f"Profit Factor: {best['profit_factor']:.2f}")
print(f"Avg Win: {best['avg_win']*100:.1f}%")
print(f"Avg Loss: {best['avg_loss']*100:.1f}%")

---
## Detailed Trade Analysis - Best Strategy

In [ ]:
# Run best strategy
best_trades = backtest_trailing_stop(
    close=close,
    entries=entries,
    stop_loss=best['stop_loss'],
    trailing_stop=best['trail_stop'],
    min_profit_to_trail=best['min_profit'],
    max_hold_days=180
)

print(f"\nTRADE DETAILS - {best['strategy']}")
print("="*100)

display_trades = best_trades.copy()
display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
display_trades['entry_price'] = display_trades['entry_price'].round(0).astype(int)
display_trades['exit_price'] = display_trades['exit_price'].round(0).astype(int)
display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)
display_trades['peak_pnl_pct'] = (display_trades['peak_pnl_pct'] * 100).round(1)

print(display_trades[['entry_date', 'entry_price', 'exit_date', 'exit_price', 
                      'pnl_pct', 'peak_pnl_pct', 'days_held', 'exit_reason']].to_string(index=False))

In [ ]:
# Exit reason breakdown
print("\nEXIT REASON BREAKDOWN")
print("="*50)
for reason in best_trades['exit_reason'].unique():
    subset = best_trades[best_trades['exit_reason'] == reason]
    count = len(subset)
    avg_pnl = subset['pnl_pct'].mean() * 100
    print(f"{reason}: {count} trades ({count/len(best_trades)*100:.0f}%) - Avg PnL: {avg_pnl:+.1f}%")

In [ ]:
# How much of the peak did we capture?
print("\nPROFIT CAPTURE ANALYSIS")
print("="*50)
winners = best_trades[best_trades['pnl_pct'] > 0]
if len(winners) > 0:
    print(f"Winners: {len(winners)}")
    print(f"  Avg Peak Reached: {winners['peak_pnl_pct'].mean()*100:.1f}%")
    print(f"  Avg Exit PnL: {winners['pnl_pct'].mean()*100:.1f}%")
    capture_rate = winners['pnl_pct'].mean() / winners['peak_pnl_pct'].mean() if winners['peak_pnl_pct'].mean() > 0 else 0
    print(f"  Capture Rate: {capture_rate*100:.0f}% of peak")

losers = best_trades[best_trades['pnl_pct'] <= 0]
if len(losers) > 0:
    print(f"\nLosers: {len(losers)}")
    print(f"  Avg Loss: {losers['pnl_pct'].mean()*100:.1f}%")

---
## Visualize Individual Trades with Stop Levels

In [ ]:
def plot_trade_detail(trade_idx):
    """Plot a single trade showing price and stop level."""
    trade = best_trades.iloc[trade_idx]
    
    # Get price data for trade period + buffer
    start = trade['entry_date'] - pd.Timedelta(days=10)
    end = trade['exit_date'] + pd.Timedelta(days=10)
    
    mask = (close.index >= start) & (close.index <= end)
    trade_close = close[mask]
    
    fig = go.Figure()
    
    # Price
    fig.add_trace(go.Scatter(x=trade_close.index, y=trade_close, name='Price',
                             line=dict(color='blue', width=2)))
    
    # Stop level history
    if len(trade['stop_history']) > 0:
        stop_df = pd.DataFrame(trade['stop_history'])
        fig.add_trace(go.Scatter(x=stop_df['date'], y=stop_df['stop'], name='Stop Level',
                                 line=dict(color='red', width=1, dash='dash')))
    
    # Entry marker
    fig.add_trace(go.Scatter(
        x=[trade['entry_date']], y=[trade['entry_price']],
        mode='markers+text',
        marker=dict(symbol='triangle-up', size=20, color='green'),
        text=[f"BUY<br>${trade['entry_price']:,.0f}"],
        textposition='bottom center',
        name='Entry'
    ))
    
    # Exit marker
    exit_color = 'green' if trade['pnl_pct'] > 0 else 'red'
    fig.add_trace(go.Scatter(
        x=[trade['exit_date']], y=[trade['exit_price']],
        mode='markers+text',
        marker=dict(symbol='triangle-down', size=20, color=exit_color),
        text=[f"EXIT<br>${trade['exit_price']:,.0f}<br>{trade['pnl_pct']*100:+.1f}%"],
        textposition='top center',
        name='Exit'
    ))
    
    fig.update_layout(
        title=f"Trade {trade_idx+1}: {trade['entry_date'].strftime('%Y-%m-%d')} → {trade['exit_date'].strftime('%Y-%m-%d')}<br>"
              f"<sup>PnL: {trade['pnl_pct']*100:+.1f}% | Peak: {trade['peak_pnl_pct']*100:.1f}% | Exit: {trade['exit_reason']}</sup>",
        yaxis_title='Price ($)',
        height=500,
        showlegend=True
    )
    return fig

In [ ]:
# Show a few example trades
print("Example Winning Trades:")
winners_idx = best_trades[best_trades['pnl_pct'] > 0].index[:3].tolist()
for idx in winners_idx:
    fig = plot_trade_detail(idx)
    fig.show()

In [ ]:
print("Example Losing Trades:")
losers_idx = best_trades[best_trades['pnl_pct'] <= 0].index[:2].tolist()
for idx in losers_idx:
    fig = plot_trade_detail(idx)
    fig.show()

---
## Full Period Overview

In [ ]:
# All trades on price chart
fig = go.Figure()

# Price
fig.add_trace(go.Scatter(x=close.index, y=close, name='BTC Price',
                         line=dict(color='lightblue', width=1)))

# Each trade as a line
for _, trade in best_trades.iterrows():
    color = 'green' if trade['pnl_pct'] > 0 else 'red'
    width = 3 if abs(trade['pnl_pct']) > 0.20 else 2  # Thicker for big moves
    
    fig.add_trace(go.Scatter(
        x=[trade['entry_date'], trade['exit_date']],
        y=[trade['entry_price'], trade['exit_price']],
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=8),
        showlegend=False,
        hovertemplate=f"Entry: {trade['entry_date'].strftime('%Y-%m-%d')}<br>" +
                      f"Exit: {trade['exit_date'].strftime('%Y-%m-%d')}<br>" +
                      f"PnL: {trade['pnl_pct']*100:.1f}%<br>" +
                      f"Peak: {trade['peak_pnl_pct']*100:.1f}%<br>" +
                      f"Exit: {trade['exit_reason']}<extra></extra>"
    ))

fig.update_layout(
    title=f"All Trades - {best['strategy']}<br><sup>Green=Win, Red=Loss. Hover for details.</sup>",
    yaxis_title='Price ($)',
    yaxis_type='log',
    height=600
)
fig.show()

In [ ]:
# Equity curve
equity = 100000 * (1 + best_trades['pnl_pct']).cumprod()
equity_df = pd.DataFrame({'date': best_trades['exit_date'], 'equity': equity.values})

# Buy and hold
bh_final = 100000 * (close.iloc[-1] / close.iloc[0])

fig = go.Figure()

fig.add_trace(go.Scatter(x=equity_df['date'], y=equity_df['equity'],
                         mode='lines+markers', name='Strategy'))
fig.add_hline(y=bh_final, line_dash='dash', line_color='gray',
              annotation_text=f'Buy & Hold: ${bh_final:,.0f}')
fig.add_hline(y=100000, line_dash='dot', line_color='black',
              annotation_text='Start: $100,000')

fig.update_layout(
    title=f'Equity Curve - {best["strategy"]}',
    yaxis_title='Portfolio Value ($)',
    height=500
)
fig.show()

print(f"\nFinal Equity: ${equity.iloc[-1]:,.0f}")
print(f"Buy & Hold: ${bh_final:,.0f}")
print(f"Excess: ${equity.iloc[-1] - bh_final:+,.0f}")

---
## Walk-Forward Validation

In [ ]:
# Walk-forward test
TRAIN_DAYS = 365
TEST_DAYS = 90
STEP_DAYS = 90

wf_results = []

total_days = len(close)
n_folds = (total_days - TRAIN_DAYS) // STEP_DAYS

for fold in range(n_folds):
    test_start = TRAIN_DAYS + fold * STEP_DAYS
    test_end = min(test_start + TEST_DAYS, total_days)
    
    if test_end <= test_start:
        break
    
    test_close = close.iloc[test_start:test_end]
    test_entries = entries.iloc[test_start:test_end]
    
    # Run strategy
    trades = backtest_trailing_stop(
        close=test_close,
        entries=test_entries,
        stop_loss=best['stop_loss'],
        trailing_stop=best['trail_stop'],
        min_profit_to_trail=best['min_profit'],
        max_hold_days=180
    )
    
    if len(trades) > 0:
        strat_return = (1 + trades['pnl_pct']).prod() - 1
        n_trades = len(trades)
    else:
        strat_return = 0
        n_trades = 0
    
    hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
    
    wf_results.append({
        'fold': fold,
        'period': close.index[test_start].strftime('%Y-%m'),
        'n_trades': n_trades,
        'strat_return': strat_return,
        'hold_return': hold_return,
        'excess': strat_return - hold_return,
        'beat_hold': strat_return > hold_return
    })
    
    status = '✓' if strat_return > hold_return else '✗'
    print(f"Fold {fold:2d}: {close.index[test_start].strftime('%Y-%m')} | "
          f"{n_trades:2d} trades | "
          f"Strat: {strat_return*100:+6.1f}% | "
          f"B&H: {hold_return*100:+6.1f}% | {status}")

wf_df = pd.DataFrame(wf_results)

In [ ]:
# Walk-forward summary
print("\n" + "="*60)
print("WALK-FORWARD SUMMARY")
print("="*60)

wf_with_trades = wf_df[wf_df['n_trades'] > 0]

print(f"\n{'Metric':<35} {'Value':>15}")
print("-"*55)
print(f"{'Total Folds':<35} {len(wf_df):>15}")
print(f"{'Folds with Trades':<35} {len(wf_with_trades):>15}")
print(f"{'Avg Strategy Return':<35} {wf_df['strat_return'].mean()*100:>14.1f}%")
print(f"{'Avg Buy&Hold Return':<35} {wf_df['hold_return'].mean()*100:>14.1f}%")
print(f"{'Avg Excess Return':<35} {wf_df['excess'].mean()*100:>+14.1f}%")
print(f"{'Beat Buy&Hold Rate':<35} {wf_df['beat_hold'].mean()*100:>14.1f}%")

if wf_with_trades['beat_hold'].mean() > 0.5:
    verdict = "✓ STRATEGY WORKS"
elif wf_with_trades['beat_hold'].mean() > 0.4:
    verdict = "~ MARGINAL"
else:
    verdict = "✗ DOESN'T BEAT BUY & HOLD"

print(f"\n🎯 VERDICT: {verdict}")

---
## Final Summary

In [ ]:
print("\n" + "="*70)
print("SOPR CAPITULATION + TRAILING STOP - FINAL RESULTS")
print("="*70)

print(f"\n📊 STRATEGY")
print(f"   Entry: SOPR < 1 AND STH SOPR < 1")
print(f"   Stop Loss: {best['stop_loss']*100:.0f}% below entry")
print(f"   Trailing Stop: {best['trail_stop']*100:.0f}% below peak")
print(f"   Trail activates after: {best['min_profit']*100:.0f}% profit")

print(f"\n📈 IN-SAMPLE")
print(f"   Total Return: {best['total_return']*100:.1f}%")
print(f"   Win Rate: {best['win_rate']*100:.0f}%")
print(f"   Profit Factor: {best['profit_factor']:.2f}")
print(f"   Avg Win: {best['avg_win']*100:.1f}%")
print(f"   Avg Loss: {best['avg_loss']*100:.1f}%")

print(f"\n🔍 WALK-FORWARD")
print(f"   Beat Buy&Hold: {wf_df['beat_hold'].mean()*100:.0f}%")
print(f"   Avg Excess: {wf_df['excess'].mean()*100:+.1f}%")

print("\n" + "="*70)

In [ ]:
# Save results
import json

final_results = {
    'signal': 'sopr_double_capitulation',
    'entry': 'SOPR < 1 AND STH_SOPR < 1',
    'exit_strategy': 'trailing_stop',
    'params': {
        'stop_loss': best['stop_loss'],
        'trailing_stop': best['trail_stop'],
        'min_profit_to_trail': best['min_profit'],
        'max_hold_days': 180
    },
    'in_sample': {
        'total_return': float(best['total_return']),
        'win_rate': float(best['win_rate']),
        'profit_factor': float(best['profit_factor']),
        'avg_win': float(best['avg_win']),
        'avg_loss': float(best['avg_loss']),
        'n_trades': int(best['n_trades'])
    },
    'walk_forward': {
        'beat_hold_pct': float(wf_df['beat_hold'].mean()),
        'avg_excess': float(wf_df['excess'].mean()),
        'n_folds': len(wf_df)
    },
    'all_configs_tested': [{
        'name': r['strategy'],
        'total_return': r['total_return'],
        'win_rate': r['win_rate'],
        'profit_factor': r['profit_factor']
    } for _, r in results_df.iterrows()]
}

with open('../data/sopr_trailing_stop_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Saved to ../data/sopr_trailing_stop_results.json")